# 7. Extension points and testing

The local implementation is only one assembly. Protocols and entry points let packages contribute pipeline catalogs, input resolvers, artifact stores, and artifact indexes without coupling definitions to implementations. Execution stores and workers are replaceable through Python composition.


## Discovery entry points

A content package usually contributes standard Provium artifact/procedure catalogs plus a `provium.pipeline_catalogs` entry point containing pipeline registrations. Resolver packages use the input resolver group. Storage packages register artifact store/index factories. Discovery validates names and compatibility before accepting plugins.


In [ ]:
from provium_pipeline import (
    PIPELINE_CATALOG_ENTRY_POINT_GROUP,
    PipelineCatalog,
    discover_pipeline_catalogs,
)
from provium_pipeline.artifact import (
    ARTIFACT_INDEX_ENTRY_POINT_GROUP,
    ARTIFACT_STORE_ENTRY_POINT_GROUP,
    discover_artifact_index_factories,
    discover_artifact_store_factories,
)
from provium_pipeline.input.resolver import INPUT_RECORD_RESOLVER_ENTRY_POINT_GROUP

entry_points = {
    'pipelines': PIPELINE_CATALOG_ENTRY_POINT_GROUP,
    'resolvers': INPUT_RECORD_RESOLVER_ENTRY_POINT_GROUP,
    'stores': ARTIFACT_STORE_ENTRY_POINT_GROUP,
    'indexes': ARTIFACT_INDEX_ENTRY_POINT_GROUP,
}
discovery_functions = (
    discover_pipeline_catalogs,
    discover_artifact_store_factories,
    discover_artifact_index_factories,
)
assert len(set(entry_points.values())) == 4 and PipelineCatalog
entry_points, discovery_functions


## Choosing the right extension

- Add a **pipeline catalog** when distributing reusable graph definitions.
- Add an **input resolver** when records come from a database, manifest service, or domain query. Return deterministic records plus a result digest.
- Add an **artifact store** when bytes belong in object/cloud/network storage. Preserve staging, verification, atomic publication, materialization cleanup, and typed failures.
- Add an **artifact index** when location metadata needs a different durable backend. Preserve identity/location uniqueness and state transitions.
- Add an **execution store or worker** for distributed scheduling. Preserve compare-and-set transitions, leases, idempotency, and frozen invocation semantics.


## Testing pyramid

Test definitions and codecs as pure values. Test compilation with small fake catalogs. Test planners and transitions without processes. Run store conformance suites against every adapter. Exercise real filesystem and SQLite adapters for atomicity, reopen behavior, corruption, and races. Use Testcontainers for remote adapters. Reserve full CLI integration tests for installed entry-point discovery and end-to-end workflows.


In [ ]:
from provium_pipeline.artifact import run_artifact_store_conformance
from provium_pipeline.compatibility import require_compatible_core
from provium_pipeline.execution_store import ExecutionStore

extension_contracts = {
    'compatibility_gate': require_compatible_core,
    'execution_store': ExecutionStore,
    'artifact_store_conformance': run_artifact_store_conformance,
}
assert all(extension_contracts.values())
extension_contracts


## Current boundary before Phase 14

The implemented library is local-first: SQLite/filesystem persistence and serial CLI execution are the primary usable path. Multiprocess supervision primitives exist, and provider-neutral contracts prepare for remote storage and distributed workers, but release hardening, packaging polish, and final operational guardrails belong to Phase 14. Do not document an extension's guarantees more strongly than its conformance and integration tests prove.


## Where to go next

Re-run a small real workflow and narrate each stored object: definition, compiled meaning, snapshot, run, tasks, dispatch, attempts, output mappings, locations, and retention references. If any object still feels redundant, ask which failure or concurrency boundary would become ambiguous without it.

Reference: [extensions](../docs/extensions.md), [testing](../docs/testing.md), [operations](../docs/operations.md), and the [module map](../docs/api-reference.md). Return to the [course index](README.md).
